# Configs

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "paths.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "paths.py").is_file():
    raise FileNotFoundError("Could not locate the project root")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from paths import *
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "paths.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "paths.py").is_file():
    raise FileNotFoundError("Could not locate the project root")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from paths import *
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "paths.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "paths.py").is_file():
    raise FileNotFoundError("Could not locate the project root")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from paths import *




# Imports

In [ ]:
# Data
import os
from pathlib import Path
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import xarray as xr

# Visualization
import matplotlib.pyplot as plt
plt.style.use('ggplot')
from matplotlib import cm
import seaborn as sns
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
from matplotlib.ticker import FixedLocator

# Statistcs

from statsmodels.tsa.seasonal import seasonal_decompose
import statsmodels.api as sm

from scipy.stats import kendalltau, theilslopes
import pymannkendall as mk

# Machine Learning
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler



def classify_kendall_trend(tau, p_value):
    if p_value < 0.05 and tau > 0:
        return "increasing"
    if p_value < 0.05 and tau < 0:
        return "decreasing"
    return "no trend"


In [ ]:
SAVE_ELBOW = str(RESULTS_DIR / "clustering")



# Functions

In [ ]:
# Decompose
def decompose_series(time, df, i, period, output):

    #Temporal Time Series
    plt.rcParams["figure.figsize"] = (12,8)
    decomposition = sm.tsa.seasonal_decompose(df[i], model = 'additive', period=period)
    figures = decomposition.plot()

    figure = plt.show()

    os.makedirs(f"{output}/trends_series", exist_ok=True)

    figure = plt.savefig(f"{output}/trends_series/{i}.png")

    plt.clf()
    plt.close("all")

    return figure


def decompose_combined_plot(df, period, output, name=None):

    plt.rcParams["figure.figsize"] = (16, 12)

    # Create output directory
    os.makedirs(f"{output}/trends_series", exist_ok=True)

    # Prepare figure with 4 subplots (observed, trend, seasonal, residual)
    fig, axes = plt.subplots(4, 1, figsize=(16, 12))

    # Iterate through columns to compute decomposition and plot
    for column in df.columns:
        decomposition = sm.tsa.seasonal_decompose(df[column], model='additive', period=period)

        # Observed
        axes[0].plot(decomposition.observed, label=column)
        axes[0].set_title("Observed")
        axes[0].legend(loc="upper right", bbox_to_anchor=(1.1, 1.0))

        # Trend
        axes[1].plot(decomposition.trend, label=column)
        axes[1].set_title("Trend")
        # axes[1].legend(loc="upper right", bbox_to_anchor=(1.1, 1.0))

        # Seasonality
        axes[2].plot(decomposition.seasonal, label=column)
        axes[2].set_title("Seasonality")
        # axes[2].legend(loc="upper right", bbox_to_anchor=(1.1, 1.0))

        # Residual
        axes[3].plot(decomposition.resid, label=column)
        axes[3].set_title("Residual")
        # axes[3].legend(loc="upper right", bbox_to_anchor=(1.1, 1.0))

    # Adjust layout
    plt.tight_layout()

    # Save figure
    plt.savefig(f"{output}/trends_series/combined_decomposition - {name}.png")
    plt.clf()
    plt.close("all")


# Correlation Matrix
def correlation_matrix_half(df, method='spearman', output=None, name=None):
    """
    Generates a correlation matrix displaying only the lower triangle.

    Parameters:
    - df: DataFrame, numerical data used to compute correlations.

    Returns:
    - figure: figure object of the correlation matrix.
    """

    # Output Directory
    os.makedirs(f"{output}/matrix", exist_ok=True)

    # Matrix correlation
    matrix = df.corr(method=method)

    # Mask
    mask = np.triu(np.ones_like(matrix, dtype=bool))

    # Configs
    plt.rcParams["figure.figsize"] = (20, 20)
    sns.heatmap(
                matrix,
                mask=mask,
                annot=True,
                fmt=".2f",
                vmax=1,
                vmin=-1,
                center=0,
                cmap="RdBu_r",
                cbar_kws={"shrink": 0.8},
                )

    # Plot configs
    plt.title(f"Correlation Matrix (Lower Triangular) - {name}", fontsize=16)
    plt.tight_layout()
    plt.savefig(f"{output}/matrix/{name}.png")

    figure = plt.gcf()
    plt.show()
    plt.clf()
    plt.close("all")



In [ ]:
# Pre processing
def preprocess_data_standarizing(data, method=StandardScaler()):  # StandardScaler
    scaler = method
    scaled_data = scaler.fit_transform(data)

    return pd.DataFrame(scaled_data, index=data.index, columns=data.columns)


def preprocess_data_temporal(scaled_data):
    """
    Preprocess the data by scaling and adding temporal information.

    Parameters:
    - data: pandas DataFrame, the dataset to preprocess.

    Returns:
    - processed_data: pandas DataFrame, scaled and with temporal features added.
    """

    scaled_data['time'] = np.arange(len(scaled_data))  # Adicionar índice temporal

    return scaled_data


#### Clusterization in Time
def elbow_method(df_scaled, k_range=range(1, 11)):
    sse = []
    k_range = k_range

    # Testing values
    for k in k_range:
        kmeans = KMeans(n_clusters = k)
        kmeans.fit(df_scaled)
        sse.append(kmeans.inertia_)

    # Plot
    plt.plot(k_range, sse, 'bx-')
    plt.xlabel('k')
    plt.ylabel('Soma dos Quadrados Intra-Cluster')
    plt.title('Método do Cotovelo para Ótimo k')
    plt.show()


def perform_clustering(data, n_clusters=3):
    """
    Perform KMeans clustering with temporal continuity consideration.

    Parameters:
    - data: pandas DataFrame, the dataset to cluster.
    - n_clusters: int, number of clusters to form.

    Returns:
    - labels: array, cluster labels for each data point.
    """
    model = KMeans(n_clusters=n_clusters, random_state=42)
    labels = model.fit_predict(data)

    return labels


def perform_clustering_with_mapping_df(data, n_clusters=3):
    """
    Perform KMeans clustering and map DataFrame indices to clusters.

    Parameters:
    - data: pandas DataFrame, the dataset to cluster.
    - n_clusters: int, number of clusters to form.

    Returns:
    - labels: array, cluster labels for each data point.
    - cluster_dict: dict, mapping of cluster IDs to time periods.
    """

    model = KMeans(n_clusters=n_clusters, random_state=42)
    labels = model.fit_predict(data)  # Clustering rows (time periods)

    # Map cluster IDs to time periods (DataFrame indices)
    cluster_dict = {}
    for cluster_id in range(n_clusters):
        cluster_dict[cluster_id] = data.index[labels == cluster_id].tolist()

    return labels, cluster_dict


def elbow_and_clustering(df_scaled, data_original_index, k_range=range(1, 11), n_clusters=None):
    """
    Runs elbow method + KMeans clustering with mapping of cluster IDs to index labels.

    Parameters:
    - df_scaled: array-like or DataFrame scaled (used for KMeans input)
    - data_original_index: DataFrame.index correspondente aos períodos originais
    - k_range: range of k to test in the elbow method
    - n_clusters: k escolhido para rodar o clustering (se None, não roda clustering)

    Returns:
    - sse: lista dos SSE para cada k testado (elbow)
    - labels: cluster labels do k escolhido (ou None se não definido)
    - cluster_dict: dicionário cluster → períodos
    """

    # ---- Elbow Method ----
    sse = []
    for k in k_range:
        model = KMeans(n_clusters=k, random_state=42)
        model.fit(df_scaled)
        sse.append(model.inertia_)

    # Plot elbow
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(list(k_range), sse, "bx-")
    ax.set_xlabel("k")
    ax.set_ylabel("Within-Cluster Sum of Squares (WCSS)")
    ax.set_title("Elbow Method")
    fig.tight_layout()

    fig.savefig(f"{SAVE_ELBOW}/elbow.png", dpi=300, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    # ---- Clustering ----
    if n_clusters is None:
        return sse, None, None

    model = KMeans(n_clusters=n_clusters, random_state=42)
    labels = model.fit_predict(df_scaled)

    # Mapear cluster → períodos
    cluster_dict = {}
    for cluster_id in range(n_clusters):
        cluster_dict[cluster_id] = list(data_original_index[labels == cluster_id])

    return sse, labels, cluster_dict



In [ ]:
# Normalize textual trend label -> USED IN mk_trend_plot()
def _normalize_trend(row) -> str:
    """
    Normalize trend label using Tau sign and boolean 'Trend' flag.
    If Tau == 0, treat as 'no trend' regardless of flag.
    """

    tau = float(row["Tau"])
    raw = str(row["Trend"]).lower()  # 'true'/'false'/'nan'/custom

    if tau == 0:
        return "no trend"

    # Already standardized?
    if raw in {"increasing", "decreasing", "no trend"}:
        return raw

    # From boolean flag: if significant, use Tau's sign; otherwise, 'no trend'
    if raw == "true":
        return "increasing" if tau > 0 else "decreasing"
    if raw == "false":
        return "decreasing" if tau < 0 else "no trend"

    # Fallback: rely on Tau sign
    return "increasing" if tau > 0 else "decreasing"


# MannKendall Test
def mann_kendall_test(data, seasonal=False):
    trends = {}

    if seasonal == False:

        for col in data.columns:
            tau, p_value = kendalltau(data.index.astype(int), data[col])
            trends[col] = {"Tau": tau, "P-value": p_value}

        return pd.DataFrame(trends).T

    else:
        for col in data.columns:
            # Seasonal Mann–Kendall (using pymannkendall)
            res = mk.seasonal_test(data[col].values, period=12)
            # res.slope is inclination of Sen; res.p is the p‑value
            trends[col] = {
                "Tau": res.slope,
                "P-value": res.p
            }

    return pd.DataFrame(trends).T


def mk_trend_plot(
                  df: pd.DataFrame,
                  mk_func,
                  seasonal: bool = False,
                  p_threshold: float = 0.05,
                  title: str = "Trend Analysis – Heatwaves",
                  ylim: tuple = (-0.3, 0.6),
                  savepath: str | None = None,
                 ):
    """
    Run a Mann–Kendall trend test on all columns of a DataFrame and plot Kendall's Tau.

    Parameters
    ----------
    df : pandas.DataFrame
        Wide-format numeric time series where each column is a variable and rows are time steps.
    mk_func : callable
        Function that computes the Mann–Kendall test on the DataFrame.
        It must return a DataFrame with at least the columns:
        - 'Tau' (Kendall's tau for each column)
        - 'P-value' (two-sided p-value for each column)
        The index of the returned DataFrame should align with df's columns.
        Example signature: mk_func(df, seasonal: bool) -> DataFrame
    seasonal : bool, optional
        Whether to apply a seasonal Mann–Kendall. Passed through to `mk_func`.
    p_threshold : float, optional
        Significance threshold to label presence/absence of trend (default 0.05).
    title : str, optional
        Plot title.
    ylim : tuple[float, float], optional
        y-axis limits for Tau visualization.
    savepath : str | None, optional
        If provided, the plot is saved to this path (e.g., "trend_plot.png").

    Returns
    -------
    fig : matplotlib.figure.Figure
        The created matplotlib figure.
    df_plot : pandas.DataFrame
        Tidy results with cleaned/derived columns:
        - 'Tau'       : Kendall's tau (NaNs filled with 0 for plotting)
        - 'P-value'   : p-values (NaNs filled with a large sentinel for plotting)
        - 'Trend'     : boolean trend flag from p_threshold
        - 'Trend_clean': normalized textual label ('increasing'/'decreasing'/'no trend')
        - 'Tau_plot'  : same as Tau after NaN handling
        Sorted by 'Tau' ascending.
    """

    # ---------- 1) Run Mann–Kendall ----------
    trends = mk_func(df, seasonal=seasonal)
    if not {"Tau", "P-value"}.issubset(trends.columns):
        raise ValueError("`mk_func` must return a DataFrame with columns: 'Tau' and 'P-value'.")

    # Align to df columns (defensive) and copy for editing
    test = trends.reindex(df.columns).copy()

    # Binary trend flag by p-value
    test["Trend"] = np.where(test["P-value"] < p_threshold, True, False)

    # ---------- 2) Prepare for plotting ----------
    df_plot = test.copy()

    # Fill NaNs to avoid plotting issues (Tau=0 => visually 'no trend')
    df_plot["Tau"] = df_plot["Tau"].fillna(0.0)
    df_plot["P-value"] = df_plot["P-value"].fillna(1e6)

    df_plot["Trend_clean"] = df_plot.apply(_normalize_trend, axis=1)
    df_plot["Tau_plot"] = df_plot["Tau"]

    # Sort by Tau for a nicer left→right progression
    df_plot = df_plot.sort_values("Tau", ascending=True)

    # ---------- 3) Style dictionaries ----------
    colors = {
              "increasing": "darkgreen",
              "decreasing": "red",
              "no trend": "gray",
             }
    markers = {
               "increasing": "o",
               "decreasing": "o",
               "no trend": "x",
              }

    # ---------- 4) Plot ----------
    fig = plt.figure(figsize=(14, 6))
    ax = plt.gca()

    # Scatter one-by-one to attach category-specific style
    for var, row in df_plot.iterrows():
        cat = row["Trend_clean"]
        ax.scatter(
                   var,
                   row["Tau_plot"],
                   color=colors.get(cat, "gray"),
                   marker=markers.get(cat, "o"),
                   s=80,
                   label=cat.capitalize(),  # temp labels: will deduplicate next
                  )

    # Horizontal reference at Tau = 0
    ax.axhline(0, color="red", linestyle="--", linewidth=1.5, alpha=0.7)

    # Legend without duplicates and in custom order
    handles, labels = ax.get_legend_handles_labels()
    uniq = dict(zip(labels, handles))
    order = ["Increasing", "No trend", "Decreasing"]
    ordered_handles = [uniq[o] for o in order if o in uniq]
    ax.legend(ordered_handles, [h.get_label() for h in ordered_handles],
              title="Trend", loc="upper left")

    # Aesthetics
    ax.set_ylabel("Kendall's Tau", fontsize=12)
    ax.set_ylim(*ylim)
    plt.xticks(rotation=45, ha="right", fontsize=10)
    ax.set_title(title, fontsize=14, weight="bold")
    ax.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.5)
    plt.tight_layout()

    # Optional save
    if savepath is not None:
        fig.savefig(savepath, dpi=300, bbox_inches="tight")

    # Note: we return the figure instead of calling plt.show() to avoid duplicate rendering in notebooks.
    return fig, df_plot




In [ ]:
# Anual Mean
def anual_accumulated_mean(df, index_col=None):
    """
    Calculates the mean over 3 defined periods (1950–1974, 1975–1998, 1999–2023).

    Parameters:
        df (pd.DataFrame): DataFrame containing the relevant data (numeric columns).
        index_col (str, optional): Column to be used as the datetime index.
                                If None, assumes the index is already datetime.

    Returns:
        pd.DataFrame: DataFrame with 3 rows, representing the mean values for each period.
    """


    # Definir os períodos
    P1 = df.loc["1950-01-01":"1974-12-31"].resample('YS').sum().mean()
    P2 = df.loc["1975-01-01":"1998-12-31"].resample('YS').sum().mean()
    P3 = df.loc["1999-01-01":"2023-12-31"].resample('YS').sum().mean()

    # Criar DataFrame de saída
    resultado = pd.DataFrame({
                             "Período": ["1950-1974", "1975-1998", "1999-2023"],
                             **{col: [P1[col], P2[col], P3[col]] for col in df.columns}
                             })

    return resultado


def plot_bar_timeseries(df, date_col, value_col, figsize=(12,5)):
    plt.figure(figsize=figsize)
    ax = sns.barplot(data=df, x=date_col, y=value_col)

    for p in ax.patches:
        value = p.get_height()
        ax.annotate(
                    f"{int(round(value))}",
                    (p.get_x() + p.get_width() / 2, value),
                    ha='center', va='bottom'
                   )

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


def plot_timeseries(df, value_col, figsize=(12,5)):
    plt.figure(figsize=figsize)
    plt.plot(df.index, df[value_col])
    plt.xlabel("Time")
    plt.title(value_col)
    plt.ylim(0, 200)
    plt.tight_layout()
    plt.show()



# MannKendall Daily Tables

In [ ]:
capitals_list = [
                "Porto Velho",
                "Rio Branco",
                "Manaus",
                "Boa Vista",
                "Belém",
                "Macapá",
                "Palmas",

                "São Luís",
                "Teresina",
                "Fortaleza",
                "Natal",
                "João Pessoa",
                "Recife",
                "Maceió",
                "Aracaju",
                "Salvador",

                "Belo Horizonte",
                "Vitória",
                "Rio de Janeiro",
                "São Paulo",

                "Curitiba",
                "Florianópolis",
                "Porto Alegre",

                "Campo Grande",
                "Cuiabá",
                "Goiânia",
                "Brasília",
                ]

capitals_dict = {
                 "Porto Velho": "RO",
                 "Rio Branco": "AC",
                 "Manaus": "AM",
                 "Boa Vista": "RR",
                 "Belém": "PA",
                 "Macapá": "AP",
                 "Palmas": "TO",

                 "São Luís": "MA",
                 "Teresina": "PI",
                 "Fortaleza": "CE",
                 "Natal": "RN",
                 "João Pessoa": "PB",
                 "Recife": "PE",
                 "Maceió": "AL",
                 "Aracaju": "SE",
                 "Salvador": "BA",

                 "Belo Horizonte": "MG",
                 "Vitória": "ES",
                 "Rio de Janeiro": "RJ",
                 "São Paulo": "SP",

                 "Curitiba": "PR",
                 "Florianópolis": "SC",
                 "Porto Alegre": "RS",

                 "Campo Grande": "MS",
                 "Cuiabá": "MT",
                 "Goiânia": "GO",
                 "Brasília": "DF",
                 }



In [ ]:
indices_cols = [
                "Ind_Prod_Diary",
                "NOAA_HWI_mean",
                "NOAA_HWI_max",
                "NOAA_HWI_min",
                "anomaly_max",
                ]



In [ ]:
# Base Path
base_path = str(ERA5_MONTHLY_DIR)
rows = []



In [ ]:
for capital in capitals_list:
    uf = capitals_dict[capital]

    path = os.path.join(
                        ERA5_DAILY_DIR,
                        capital,
                        f"{capital}_heatwave_ERA5_diary.parquet"
                        )

    df = pd.read_parquet(path)

    df.set_index("time", inplace=True)

    result = {"UF": uf, "capital": capital}

    for col in indices_cols:
        s = df[col].dropna()
        tau, p_value = kendalltau(np.arange(len(s)), s.values)

        result[f"{col}_tau"] = tau
        result[f"{col}_pvalue"] = p_value
        result[f"{col}_trend"] = classify_kendall_trend(tau, p_value)  # 'increasing', 'decreasing', 'no trend'

    rows.append(result)

df_trends = pd.DataFrame(rows).set_index("UF")
df_trends



In [ ]:
output_path = str(RESULTS_DIR / "trend_analysis")
df_trends.to_csv(f'{output_path}/mannkendall_indices_diary.csv')



# Mann-Kendall Monthly Table

In [ ]:
capitals_list = [
                "Porto Velho",
                "Rio Branco",
                "Manaus",
                "Boa Vista",
                "Belém",
                "Macapá",
                "Palmas",

                "São Luís",
                "Teresina",
                "Fortaleza",
                "Natal",
                "João Pessoa",
                "Recife",
                "Maceió",
                "Aracaju",
                "Salvador",

                "Belo Horizonte",
                "Vitória",
                "Rio de Janeiro",
                "São Paulo",

                "Curitiba",
                "Florianópolis",
                "Porto Alegre",

                "Campo Grande",
                "Cuiabá",
                "Goiânia",
                "Brasília",
                ]

capitals_dict = {
                 "Porto Velho": "RO",
                 "Rio Branco": "AC",
                 "Manaus": "AM",
                 "Boa Vista": "RR",
                 "Belém": "PA",
                 "Macapá": "AP",
                 "Palmas": "TO",

                 "São Luís": "MA",
                 "Teresina": "PI",
                 "Fortaleza": "CE",
                 "Natal": "RN",
                 "João Pessoa": "PB",
                 "Recife": "PE",
                 "Maceió": "AL",
                 "Aracaju": "SE",
                 "Salvador": "BA",

                 "Belo Horizonte": "MG",
                 "Vitória": "ES",
                 "Rio de Janeiro": "RJ",
                 "São Paulo": "SP",

                 "Curitiba": "PR",
                 "Florianópolis": "SC",
                 "Porto Alegre": "RS",

                 "Campo Grande": "MS",
                 "Cuiabá": "MT",
                 "Goiânia": "GO",
                 "Brasília": "DF",
                 }



In [ ]:
indices_cols = [
                'ind_prod_monthly_qtd',
                'noaa_qtd_monthly',
                'omm_qtd_monthly'
                ]



In [ ]:
# Base Path
base_path = str(ERA5_MONTHLY_DIR)
rows = []



In [ ]:
# Qtd days with heatwaves conditions

for capital in capitals_list:
    uf = capitals_dict[capital]

    path = os.path.join(
                        base_path,
                        capital,
                        f"{capital}_heatwave_ERA5_monthly.parquet"
                        )

    df = pd.read_parquet(path)

    df.set_index("time", inplace=True)

    result = {"UF": uf, "capital": capital}

    for col in indices_cols:
        s = df[col].dropna()
        tau, p_value = kendalltau(np.arange(len(s)), s.values)

        result[f"{col}_tau"] = tau
        result[f"{col}_pvalue"] = p_value
        result[f"{col}_trend"] = classify_kendall_trend(tau, p_value)  # 'increasing', 'decreasing', 'no trend'

    rows.append(result)

df_trends = pd.DataFrame(rows).set_index("UF")
df_trends



In [ ]:
output_path = str(RESULTS_DIR / "trend_analysis")
df_trends.to_csv(f'{output_path}/mannkendall_indices_qtd_monthly.csv')



In [ ]:
# Heatwaves 3 episodes
rows = []

indices_cols = [
                'ind_prod_3episodes',
                'noaa_3episodes',
                'omm_3episodes'
                ]

for capital in capitals_list:
    uf = capitals_dict[capital]

    path = os.path.join(
                        base_path,
                        capital,
                        f"{capital}_heatwave_ERA5_monthly.parquet"
                        )

    df = pd.read_parquet(path)

    df.set_index("time", inplace=True)

    result = {"UF": uf, "capital": capital}

    for col in indices_cols:
        s = df[col].dropna()
        tau, p_value = kendalltau(np.arange(len(s)), s.values)

        result[f"{col}_tau"] = tau
        result[f"{col}_pvalue"] = p_value
        result[f"{col}_trend"] = classify_kendall_trend(tau, p_value)  # 'increasing', 'decreasing', 'no trend'

    rows.append(result)

df_trends = pd.DataFrame(rows).set_index("UF")
display(df_trends)

output_path = str(RESULTS_DIR / "trend_analysis")
df_trends.to_csv(f'{output_path}/mannkendall_indices_3episodes_monthly.csv')



In [ ]:
# Heatwaves 5 episodes
rows = []

indices_cols = [
                'ind_prod_5episodes',
                'noaa_5episodes',
                'omm_5episodes'
                ]

for capital in capitals_list:
    uf = capitals_dict[capital]

    path = os.path.join(
                        base_path,
                        capital,
                        f"{capital}_heatwave_ERA5_monthly.parquet"
                        )

    df = pd.read_parquet(path)

    df.set_index("time", inplace=True)

    result = {"UF": uf, "capital": capital}

    for col in indices_cols:
        s = df[col].dropna()
        tau, p_value = kendalltau(np.arange(len(s)), s.values)

        result[f"{col}_tau"] = tau
        result[f"{col}_pvalue"] = p_value
        result[f"{col}_trend"] = classify_kendall_trend(tau, p_value)  # 'increasing', 'decreasing', 'no trend'

    rows.append(result)

df_trends = pd.DataFrame(rows).set_index("UF")

display(df_trends)

output_path = str(RESULTS_DIR / "trend_analysis")
df_trends.to_csv(f'{output_path}/mannkendall_indices_5episodes_monthly.csv')



# Clustering

In [ ]:
capitals_list = [
                "Porto Velho",
                "Rio Branco",
                "Manaus",
                "Boa Vista",
                "Belém",
                "Macapá",
                "Palmas",

                "São Luís",
                "Teresina",
                "Fortaleza",
                "Natal",
                "João Pessoa",
                "Recife",
                "Maceió",
                "Aracaju",
                "Salvador",

                "Belo Horizonte",
                "Vitória",
                "Rio de Janeiro",
                "São Paulo",

                "Curitiba",
                "Florianópolis",
                "Porto Alegre",

                "Campo Grande",
                "Cuiabá",
                "Goiânia",
                "Brasília",
                ]

capitals_dict = {
                 "Porto Velho": "RO",
                 "Rio Branco": "AC",
                 "Manaus": "AM",
                 "Boa Vista": "RR",
                 "Belém": "PA",
                 "Macapá": "AP",
                 "Palmas": "TO",

                 "São Luís": "MA",
                 "Teresina": "PI",
                 "Fortaleza": "CE",
                 "Natal": "RN",
                 "João Pessoa": "PB",
                 "Recife": "PE",
                 "Maceió": "AL",
                 "Aracaju": "SE",
                 "Salvador": "BA",

                 "Belo Horizonte": "MG",
                 "Vitória": "ES",
                 "Rio de Janeiro": "RJ",
                 "São Paulo": "SP",

                 "Curitiba": "PR",
                 "Florianópolis": "SC",
                 "Porto Alegre": "RS",

                 "Campo Grande": "MS",
                 "Cuiabá": "MT",
                 "Goiânia": "GO",
                 "Brasília": "DF",
                 }

indices_cols = [
                'Ind_Prod_Monthly',
                'ind_prod_monthly_qtd',
                'ind_prod_3episodes',
                'ind_prod_5episodes',

                'noaa_qtd_monthly',
                'noaa_3episodes',
                'noaa_5episodes',

                'omm_qtd_monthly',
                'omm_3episodes',
                'omm_5episodes'
                ]



In [ ]:
base_path = str(ERA5_MONTHLY_DIR)
dfs = []



In [ ]:
for capital in capitals_list:
    uf = capitals_dict[capital]

    path = os.path.join(
                        base_path,
                        capital,
                        f"{capital}_heatwave_ERA5_monthly.parquet"
                        )

    df = pd.read_parquet(path)

    df = df.set_index("time")[indices_cols]

    df = df.add_suffix(f"_{uf}")

    dfs.append(df)


df_final = pd.concat(dfs, axis=1).sort_index()



In [ ]:
output_path = str(RESULTS_DIR / "trend_analysis")
df_final.to_parquet(CLUSTERING_METRICS_FILE)



## Clustering Process in Time Series

In [ ]:
pth = str(CLUSTERING_METRICS_FILE)
df = pd.read_parquet(pth).loc[:'2023-12-01']
display(df.info())
display(df)



In [ ]:
# Cluster Periods
df_xhwi_scaled = preprocess_data_standarizing(df)
df_xhwi_scaled = preprocess_data_temporal(df_xhwi_scaled)
labels, cluster_dict = perform_clustering_with_mapping_df(df_xhwi_scaled, n_clusters=3)

# Exibir resultados
print("Dicionário de Clusters:")
for cluster, indices in cluster_dict.items():
    print(f"Cluster {cluster}: {indices}")



In [ ]:
# Aqui Lívia
df_xhwi_scaled = preprocess_data_standarizing(df)
df_xhwi_scaled = preprocess_data_temporal(df_xhwi_scaled)
_, labels, cluster_dict = elbow_and_clustering(df_xhwi_scaled, df_xhwi_scaled.index, n_clusters=3)

# Exibir resultados
print("Dicionário de Clusters:")
for cluster, indices in cluster_dict.items():
    print(f"Cluster {cluster}: {indices}")



In [ ]:
df_series = df_xhwi_scaled.iloc[:, :-1]
ufs = df_series.columns.str[-2:]
df_mean = df_series.groupby(ufs, axis=1).median()

# Plot
ax = df_mean.plot(figsize=(16, 10))
ax.legend(
          loc="center right",
          bbox_to_anchor=(1.05, 0.5)
          )

plt.tight_layout()
plt.show()



In [ ]:
df_xhwi_scaled.iloc[:, :-1]



In [ ]:
df_series = df_xhwi_scaled.iloc[:, :-1]
df_mean = df_series.mean(axis=1)
df_mean.index = pd.to_datetime(df_mean.index)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

# Plot the mean standardized signal as a single time series
df_mean.plot(ax=ax, color="blue", label="Mean across metrics of all city capitals")
df_mean.rolling(120, center=True).mean().plot(ax=ax, color="red", linewidth=2, label="120-month rolling mean")

# Add reference vertical dashed lines at key years
ax.axvline(pd.Timestamp("1975-01-01"), color="k", linestyle="--", linewidth=1.5)
ax.axvline(pd.Timestamp("1999-01-01"), color="k", linestyle="--", linewidth=1.5)

# Set title and axis label
ax.set_title(
             "Mean heatwave-metric signal across Brazilian state capitals (standardized metrics)",
             pad=12
             )
ax.set_xlabel("Year")

# Periods
y_top = 0.97
ax.text(0.02, y_top, r"$\bf{P1}$", transform=ax.transAxes, ha="center", va="top", color="k", fontsize=14)
ax.text(0.36, y_top, r"$\bf{P2}$", transform=ax.transAxes, ha="center", va="top", color="k", fontsize=14)
ax.text(0.68, y_top, r"$\bf{P3}$", transform=ax.transAxes, ha="center", va="top", color="k", fontsize=14)

# Force a draw so tick labels exist (needed to copy tick font properties)
fig.canvas.draw()

# Match the annotation style to the x-axis tick labels
tick_fs = ax.xaxis.get_ticklabels()[0].get_size()
tick_color = ax.xaxis.get_ticklabels()[0].get_color()

# Use the bottom y-limit as anchor and shift text downward by a fixed offset (points)
ymin, _ = ax.get_ylim()

for date_str, label in [("1950-01-01", "1950"), ("1975-01-01", "1975"), ("2023-12-01", "2023")]:
    ax.annotate(
                label,
                xy=(pd.Timestamp(date_str), ymin),   # anchor at the bottom of the axes
                xytext=(0, -7),                     # shift downward (increase magnitude to go lower)
                textcoords="offset points",
                ha="center",
                va="top",
                fontsize=tick_fs,
                color=tick_color
                )

# Legend
ax.legend(loc="upper right")

# Layout and export
fig.tight_layout()
os.makedirs(SAVE_ELBOW, exist_ok=True)
fig.savefig(
            f"{SAVE_ELBOW}/mean_heatwave_metrics_capitals.png",
            dpi=300,
            bbox_inches="tight"
            )

plt.show()
plt.close(fig)



In [ ]:
df_xhwi_scaled.iloc[:, 70:-1]



In [ ]:
df_series_full = df_xhwi_scaled.iloc[:, :-1]
df_full_mean = df_series_full.mean(axis=1)

df_series = df_xhwi_scaled.iloc[:, 70:-1]
df_mean = df_series.mean(axis=1)
df_mean.index = pd.to_datetime(df_mean.index)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

# Plot the mean standardized signal as a single time series
df_full_mean.plot(ax=ax, color="gray", linestyle="--", label="Mean across metrics of all city capitals")
df_mean.plot(ax=ax, color="blue", label="Mean across metrics for all city capitals except those in the North Region")
df_mean.rolling(120, center=True).mean().plot(ax=ax, color="red", linewidth=2, label="120-month rolling mean")

# Add reference vertical dashed lines at key years
ax.axvline(pd.Timestamp("1975-01-01"), color="k", linestyle="--", linewidth=1.5)
ax.axvline(pd.Timestamp("1999-01-01"), color="k", linestyle="--", linewidth=1.5)

# Set title and axis label
ax.set_title(
             "Mean heatwave-metric signal across Brazilian state capitals (standardized metrics)",
             pad=12
             )
ax.set_xlabel("Year")

# Periods
y_top = 0.97
ax.text(0.02, y_top, r"$\bf{P1}$", transform=ax.transAxes, ha="center", va="top", color="k", fontsize=14)
ax.text(0.36, y_top, r"$\bf{P2}$", transform=ax.transAxes, ha="center", va="top", color="k", fontsize=14)
ax.text(0.68, y_top, r"$\bf{P3}$", transform=ax.transAxes, ha="center", va="top", color="k", fontsize=14)

# Force a draw so tick labels exist (needed to copy tick font properties)
fig.canvas.draw()

# Match the annotation style to the x-axis tick labels
tick_fs = ax.xaxis.get_ticklabels()[0].get_size()
tick_color = ax.xaxis.get_ticklabels()[0].get_color()

# Use the bottom y-limit as anchor and shift text downward by a fixed offset (points)
ymin, _ = ax.get_ylim()

for date_str, label in [("1950-01-01", "1950"), ("1975-01-01", "1975"), ("2023-12-01", "2023")]:
    ax.annotate(
                label,
                xy=(pd.Timestamp(date_str), ymin),   # anchor at the bottom of the axes
                xytext=(0, -7),                     # shift downward (increase magnitude to go lower)
                textcoords="offset points",
                ha="center",
                va="top",
                fontsize=tick_fs,
                color=tick_color
                )

# Legend
ax.legend(loc="upper right")

# Layout and export
fig.tight_layout()
os.makedirs(SAVE_ELBOW, exist_ok=True)
fig.savefig(
            f"{SAVE_ELBOW}/mean_heatwave_metrics_test_capitals.png",
            dpi=300,
            bbox_inches="tight"
            )

plt.show()
plt.close(fig)



In [ ]:
df_mean.describe()



In [ ]:
p1_out = df_mean.loc['1950-01-01':'1974-12-01']
top6 = p1_out.nlargest(6)
top6



In [ ]:
pd.set_option("display.max_rows", None)
df_p1 = df.loc['1950-01-01':'1974-12-01']
df_p1.loc["1963-10-01"]



In [ ]:
# Anual Accumulated Mean
df_aam_xhwi = anual_accumulated_mean(df)

display(df_aam_xhwi)
#df_aam_xhwi.to_csv('PROJECT_ROOT/results/oficial_results/anual_accumulated_mean.csv')

# % Difference
df_diff_xhwi = df_aam_xhwi.iloc[:, 1:].pct_change(axis=0) * 100
df_diff_xhwi = df_diff_xhwi.iloc[1:, :]

pct_P3_P1 = (((df_aam_xhwi.iloc[2, 1:] - df_aam_xhwi.iloc[0, 1:]) / df_aam_xhwi.iloc[0, 1:]) * 100).T

df_diff_xhwi['Período'] = ''
df_diff_xhwi = df_diff_xhwi[ ["Período"] + [col for col in df_diff_xhwi.columns if col != "Período"] ]
df_diff_xhwi.loc[1, 'Período'] = 'P1_P2'
df_diff_xhwi.loc[2, 'Período'] = 'P2_P3'

display(df_diff_xhwi)

pct_P3_P1.name = 'P1_P3'
pct_P3_P1 = pct_P3_P1.to_frame()
pct_P3_P1 = pct_P3_P1.T.copy()
pct_P3_P1.reset_index(inplace=True)
pct_P3_P1.rename(columns={pct_P3_P1.columns[0]: "Período"}, inplace=True)
display(pct_P3_P1)



In [ ]:
pct_p_p = pd.concat([df_diff_xhwi, pct_P3_P1], axis=0)
pct_p_p.reset_index(inplace=True)
pct_p_p.drop(columns=["index"], inplace=True)
pct_p_p



In [ ]:
pct_p_p.set_index("Período", inplace=True)



In [ ]:
pct_p_p = pct_p_p.apply(pd.to_numeric, errors="coerce")

pct_p_p = pct_p_p.replace([np.inf, -np.inf], np.nan)
pct_p_p



In [ ]:
pct_p_p



In [ ]:
# Coloring accordingly to percentil

vals = pct_p_p.to_numpy().ravel()
vals = vals[~np.isnan(vals)]

p20, p40, p60, p80 = np.percentile(vals, [20, 40, 60, 80])

colors = [mcolors.rgb2hex(plt.cm.coolwarm(i)) for i in np.linspace(0, 1, 5)]

def color_by_percentile(v):
    if np.isnan(v):
        return ""
    if v <= p20:
        return f"background-color: {colors[0]}"
    if v <= p40:
        return f"background-color: {colors[1]}"
    if v <= p60:
        return f"background-color: {colors[2]}"
    if v <= p80:
        return f"background-color: {colors[3]}"
    return f"background-color: {colors[4]}"

styled = pct_p_p.style.map(color_by_percentile)

styled



In [ ]:
legend_df = pd.DataFrame(
    {
        "Percentile range": [
            "Not a Number",
            "≤ 20%",
            "20–40%",
            "40–60%",
            "60–80%",
            "≥ 80%",
        ],
        "Percentage difference": [
            np.nan,
            p20,
            (p20 + p40) / 2,
            (p40 + p60) / 2,
            (p60 + p80) / 2,
            p80 + 1e-6,  # ensures last bin
        ],
    }
).set_index("Percentile range")

legend_styled = legend_df.style.map(color_by_percentile)
legend_styled



In [ ]:
base_output = str(RESULTS_DIR / "period_comparison")
df = df_aam_xhwi.copy()

# se "Período" estiver como índice (como na imagem), traz para coluna:
if "Período" not in df.columns:
    df = df.reset_index()  # só se o índice for "Período"

# dicionário capitais → UF
capitals_dict = {
                "Porto Velho": "RO",
                "Rio Branco": "AC",
                "Manaus": "AM",
                "Boa Vista": "RR",
                "Belém": "PA",
                "Macapá": "AP",
                "Palmas": "TO",

                "São Luís": "MA",
                "Teresina": "PI",
                "Fortaleza": "CE",
                "Natal": "RN",
                "João Pessoa": "PB",
                "Recife": "PE",
                "Maceió": "AL",
                "Aracaju": "SE",
                "Salvador": "BA",

                "Belo Horizonte": "MG",
                "Vitória": "ES",
                "Rio de Janeiro": "RJ",
                "São Paulo": "SP",

                "Curitiba": "PR",
                "Florianópolis": "SC",
                "Porto Alegre": "RS",

                "Campo Grande": "MS",
                "Cuiabá": "MT",
                "Goiânia": "GO",
                "Brasília": "DF",
                }

# lista de colunas-base (sem UF)
base_cols = [
            "Ind_Prod_Monthly",
            "ind_prod_monthly_qtd",
            "ind_prod_3episodes",
            "ind_prod_5episodes",
            "noaa_qtd_monthly",
            "noaa_3episodes",
            "noaa_5episodes",
            "omm_qtd_monthly",
            "omm_3episodes",
            "omm_5episodes",
            ]

# checagem básica: todas as colunas com sufixo UF existem?
ufs = set(capitals_dict.values())
esperadas = [f"{base}_{uf}" for base in base_cols for uf in ufs]
faltando = set(esperadas) - set(df.columns)
if faltando:
    raise ValueError(f"Faltam colunas no df: {faltando}")

# mapeia período → nome do arquivo
arquivos = {
           "1950-1974": "1950_1974_indices_capitais.csv",
           "1975-1998": "1975_1998_indices_capitais.csv",
           "1999-2023": "1999_2023_indices_capitais.csv",
           }

for periodo, nome_arquivo in arquivos.items():
    # pega a linha correspondente a esse período
    linha_periodo = df.loc[df["Período"] == periodo]
    if len(linha_periodo) != 1:
        raise ValueError(f"Esperava 1 linha para o período {periodo}, encontrei {len(linha_periodo)}.")
    linha_periodo = linha_periodo.iloc[0]

    # monta a tabela nova: uma linha por capital
    linhas_saida = []
    for capital, uf in capitals_dict.items():
        linha_capital = {"capitais": capital}
        for base in base_cols:
            col_nome = f"{base}_{uf}"
            linha_capital[base] = linha_periodo[col_nome]
        linhas_saida.append(linha_capital)

    df_saida = pd.DataFrame(linhas_saida, columns=["capitais"] + base_cols)

    # salva CSV
    df_saida.to_csv(f'{base_output}/{nome_arquivo}', index=False, encoding="utf-8")
    print(f"Gerado: {nome_arquivo} ({len(df_saida)} capitais)")



In [ ]:
base_output = str(RESULTS_DIR / "period_comparison")
df = df_diff_xhwi.copy()

# se "Período" estiver como índice (como na imagem), traz para coluna:
if "Período" not in df.columns:
    df = df.reset_index()  # só se o índice for "Período"

# dicionário capitais → UF
capitals_dict = {
                "Porto Velho": "RO",
                "Rio Branco": "AC",
                "Manaus": "AM",
                "Boa Vista": "RR",
                "Belém": "PA",
                "Macapá": "AP",
                "Palmas": "TO",

                "São Luís": "MA",
                "Teresina": "PI",
                "Fortaleza": "CE",
                "Natal": "RN",
                "João Pessoa": "PB",
                "Recife": "PE",
                "Maceió": "AL",
                "Aracaju": "SE",
                "Salvador": "BA",

                "Belo Horizonte": "MG",
                "Vitória": "ES",
                "Rio de Janeiro": "RJ",
                "São Paulo": "SP",

                "Curitiba": "PR",
                "Florianópolis": "SC",
                "Porto Alegre": "RS",

                "Campo Grande": "MS",
                "Cuiabá": "MT",
                "Goiânia": "GO",
                "Brasília": "DF",
                }

# lista de colunas-base (sem UF)
base_cols = [
            "Ind_Prod_Monthly",
            "ind_prod_monthly_qtd",
            "ind_prod_3episodes",
            "ind_prod_5episodes",
            "noaa_qtd_monthly",
            "noaa_3episodes",
            "noaa_5episodes",
            "omm_qtd_monthly",
            "omm_3episodes",
            "omm_5episodes",
            ]

# checagem básica: todas as colunas com sufixo UF existem?
ufs = set(capitals_dict.values())
esperadas = [f"{base}_{uf}" for base in base_cols for uf in ufs]
faltando = set(esperadas) - set(df.columns)
if faltando:
    raise ValueError(f"Faltam colunas no df: {faltando}")

# mapeia período → nome do arquivo
arquivos = {
           "P1_P2": "P1_P2_diffpercent_capitais.csv",
           "P2_P3": "P2_P3_diffpercent_capitais.csv"
           }

for periodo, nome_arquivo in arquivos.items():
    # pega a linha correspondente a esse período
    linha_periodo = df.loc[df["Período"] == periodo]
    if len(linha_periodo) != 1:
        raise ValueError(f"Esperava 1 linha para o período {periodo}, encontrei {len(linha_periodo)}.")
    linha_periodo = linha_periodo.iloc[0]

    # monta a tabela nova: uma linha por capital
    linhas_saida = []
    for capital, uf in capitals_dict.items():
        linha_capital = {"capitais": capital}
        for base in base_cols:
            col_nome = f"{base}_{uf}"
            linha_capital[base] = linha_periodo[col_nome]
        linhas_saida.append(linha_capital)

    df_saida = pd.DataFrame(linhas_saida, columns=["capitais"] + base_cols)

    # salva CSV
    df_saida.to_csv(f'{base_output}/{nome_arquivo}', index=False, encoding="utf-8")
    print(f"Gerado: {nome_arquivo} ({len(df_saida)} capitais)")



In [ ]:
base_output = str(RESULTS_DIR / "period_comparison")
df = pct_P3_P1.copy()

# se "Período" estiver como índice (como na imagem), traz para coluna:
if "Período" not in df.columns:
    df = df.reset_index()  # só se o índice for "Período"

# dicionário capitais → UF
capitals_dict = {
                "Porto Velho": "RO",
                "Rio Branco": "AC",
                "Manaus": "AM",
                "Boa Vista": "RR",
                "Belém": "PA",
                "Macapá": "AP",
                "Palmas": "TO",

                "São Luís": "MA",
                "Teresina": "PI",
                "Fortaleza": "CE",
                "Natal": "RN",
                "João Pessoa": "PB",
                "Recife": "PE",
                "Maceió": "AL",
                "Aracaju": "SE",
                "Salvador": "BA",

                "Belo Horizonte": "MG",
                "Vitória": "ES",
                "Rio de Janeiro": "RJ",
                "São Paulo": "SP",

                "Curitiba": "PR",
                "Florianópolis": "SC",
                "Porto Alegre": "RS",

                "Campo Grande": "MS",
                "Cuiabá": "MT",
                "Goiânia": "GO",
                "Brasília": "DF",
                }

# lista de colunas-base (sem UF)
base_cols = [
            "Ind_Prod_Monthly",
            "ind_prod_monthly_qtd",
            "ind_prod_3episodes",
            "ind_prod_5episodes",
            "noaa_qtd_monthly",
            "noaa_3episodes",
            "noaa_5episodes",
            "omm_qtd_monthly",
            "omm_3episodes",
            "omm_5episodes",
            ]

# checagem básica: todas as colunas com sufixo UF existem?
ufs = set(capitals_dict.values())
esperadas = [f"{base}_{uf}" for base in base_cols for uf in ufs]
faltando = set(esperadas) - set(df.columns)
if faltando:
    raise ValueError(f"Faltam colunas no df: {faltando}")

# mapeia período → nome do arquivo
arquivos = {
           "P1_P3": "P1_P3_diffpercent_capitais.csv"
           }

for periodo, nome_arquivo in arquivos.items():
    # pega a linha correspondente a esse período
    linha_periodo = df.loc[df["Período"] == periodo]
    if len(linha_periodo) != 1:
        raise ValueError(f"Esperava 1 linha para o período {periodo}, encontrei {len(linha_periodo)}.")
    linha_periodo = linha_periodo.iloc[0]

    # monta a tabela nova: uma linha por capital
    linhas_saida = []
    for capital, uf in capitals_dict.items():
        linha_capital = {"capitais": capital}
        for base in base_cols:
            col_nome = f"{base}_{uf}"
            linha_capital[base] = linha_periodo[col_nome]
        linhas_saida.append(linha_capital)

    df_saida = pd.DataFrame(linhas_saida, columns=["capitais"] + base_cols)

    # salva CSV
    df_saida.to_csv(f'{base_output}/{nome_arquivo}', index=False, encoding="utf-8")
    print(f"Gerado: {nome_arquivo} ({len(df_saida)} capitais)")



In [ ]:
base_output = str(RESULTS_DIR / "period_comparison")
df = pct_p_p.copy()

# se "Período" estiver como índice (como na imagem), traz para coluna:
if "Período" not in df.columns:
    df = df.reset_index()  # só se o índice for "Período"

# dicionário capitais → UF
capitals_dict = {
                "Porto Velho": "RO",
                "Rio Branco": "AC",
                "Manaus": "AM",
                "Boa Vista": "RR",
                "Belém": "PA",
                "Macapá": "AP",
                "Palmas": "TO",

                "São Luís": "MA",
                "Teresina": "PI",
                "Fortaleza": "CE",
                "Natal": "RN",
                "João Pessoa": "PB",
                "Recife": "PE",
                "Maceió": "AL",
                "Aracaju": "SE",
                "Salvador": "BA",

                "Belo Horizonte": "MG",
                "Vitória": "ES",
                "Rio de Janeiro": "RJ",
                "São Paulo": "SP",

                "Curitiba": "PR",
                "Florianópolis": "SC",
                "Porto Alegre": "RS",

                "Campo Grande": "MS",
                "Cuiabá": "MT",
                "Goiânia": "GO",
                "Brasília": "DF",
                }

# lista de colunas-base (sem UF)
base_cols = [
            "Ind_Prod_Monthly",
            "ind_prod_monthly_qtd",
            "ind_prod_3episodes",
            "ind_prod_5episodes",
            "noaa_qtd_monthly",
            "noaa_3episodes",
            "noaa_5episodes",
            "omm_qtd_monthly",
            "omm_3episodes",
            "omm_5episodes",
            ]

# checagem básica: todas as colunas com sufixo UF existem?
ufs = set(capitals_dict.values())
esperadas = [f"{base}_{uf}" for base in base_cols for uf in ufs]
faltando = set(esperadas) - set(df.columns)
if faltando:
    raise ValueError(f"Faltam colunas no df: {faltando}")

# mapeia período → nome do arquivo
arquivos = {
           "P1_P2": "P1_P2_diffpercent_capitais.xlsx",
           "P2_P3": "P2_P3_diffpercent_capitais.xlsx",
           "P1_P3": "P1_P3_diffpercent_capitais.xlsx",
           }

for periodo, nome_arquivo in arquivos.items():
    # pega a linha correspondente a esse período
    linha_periodo = df.loc[df["Período"] == periodo]
    if len(linha_periodo) != 1:
        raise ValueError(f"Esperava 1 linha para o período {periodo}, encontrei {len(linha_periodo)}.")
    linha_periodo = linha_periodo.iloc[0]

    # monta a tabela nova: uma linha por capital
    linhas_saida = []
    for capital, uf in capitals_dict.items():
        linha_capital = {"capitais": capital}
        for base in base_cols:
            col_nome = f"{base}_{uf}"
            linha_capital[base] = linha_periodo[col_nome]
        linhas_saida.append(linha_capital)

    df_saida = pd.DataFrame(linhas_saida, columns=["capitais"] + base_cols)

    # salva xlsx
    styled = df_saida.style.map(color_by_percentile, subset=df_saida.columns.difference(["capitais"]))
    styled.to_excel(f'{base_output}/{nome_arquivo}', index=False, engine="openpyxl")
    print(f"Gerado: {nome_arquivo} ({len(df_saida)} capitais)")



# Correlation with deaths

In [ ]:
capitals_list = [
                "Porto Velho",
                "Rio Branco",
                "Manaus",
                "Boa Vista",
                "Belém",
                "Macapá",
                "Palmas",

                "São Luís",
                "Teresina",
                "Fortaleza",
                "Natal",
                "João Pessoa",
                "Recife",
                "Maceió",
                "Aracaju",
                "Salvador",

                "Belo Horizonte",
                "Vitória",
                "Rio de Janeiro",
                "São Paulo",

                "Curitiba",
                "Florianópolis",
                "Porto Alegre",

                "Campo Grande",
                "Cuiabá",
                "Goiânia",
                "Brasília",
                ]

capitals_dict = {
                 "Porto Velho": "RO",
                 "Rio Branco": "AC",
                 "Manaus": "AM",
                 "Boa Vista": "RR",
                 "Belém": "PA",
                 "Macapá": "AP",
                 "Palmas": "TO",

                 "São Luís": "MA",
                 "Teresina": "PI",
                 "Fortaleza": "CE",
                 "Natal": "RN",
                 "João Pessoa": "PB",
                 "Recife": "PE",
                 "Maceió": "AL",
                 "Aracaju": "SE",
                 "Salvador": "BA",

                 "Belo Horizonte": "MG",
                 "Vitória": "ES",
                 "Rio de Janeiro": "RJ",
                 "São Paulo": "SP",

                 "Curitiba": "PR",
                 "Florianópolis": "SC",
                 "Porto Alegre": "RS",

                 "Campo Grande": "MS",
                 "Cuiabá": "MT",
                 "Goiânia": "GO",
                 "Brasília": "DF",
                 }

pop_dict = {
           "pop_RO": 460434,     # Porto Velho
           "pop_AC": 364756,     # Rio Branco
           "pop_AM": 2063689,    # Manaus
           "pop_RR": 413486,     # Boa Vista
           "pop_PA": 1303403,    # Belém
           "pop_AP": 442933,     # Macapá
           "pop_TO": 302692,     # Palmas

           "pop_MA": 1037775,    # São Luís
           "pop_PI": 866300,     # Teresina
           "pop_CE": 2428708,    # Fortaleza
           "pop_RN": 751300,     # Natal
           "pop_PB": 833932,     # João Pessoa
           "pop_PE": 1488920,    # Recife
           "pop_AL": 957916,     # Maceió
           "pop_SE": 602757,     # Aracaju
           "pop_BA": 2417678,    # Salvador

           "pop_MG": 2315560,    # Belo Horizonte
           "pop_ES": 322869,     # Vitória
           "pop_RJ": 6211223,    # Rio de Janeiro
           "pop_SP": 11451999,   # São Paulo

           "pop_PR": 1773718,    # Curitiba
           "pop_SC": 537211,     # Florianópolis
           "pop_RS": 1332845,    # Porto Alegre

           "pop_MS": 898100,     # Campo Grande
           "pop_MT": 650877,     # Cuiabá
           "pop_GO": 1437366,    # Goiânia
           "pop_DF": 2817381     # Brasília
           }

base_cols = [
            "Ind_Prod_Monthly",
            "ind_prod_monthly_qtd",
            "ind_prod_3episodes",
            "ind_prod_5episodes",
            "noaa_qtd_monthly",
            "noaa_3episodes",
            "noaa_5episodes",
            "omm_qtd_monthly",
            "omm_3episodes",
            "omm_5episodes",
            ]



In [ ]:
pth = str(CLUSTERING_METRICS_FILE)
df_hi = pd.read_parquet(pth)
df_hi = df_hi.loc['1996-01-01':'2023-12-01']
df_hi



In [ ]:
# Deaths
path_deaths = str(HEALTH_DEATHS_FILE)
df_deaths = pd.read_csv(path_deaths, sep=';')
df_deaths['time'] = pd.to_datetime(df_deaths['time'])
df_deaths.set_index('time', inplace=True)
df_deaths.index = pd.date_range(start="1996-01-01", periods=len(df_deaths), freq="MS")

df_per100k = pd.DataFrame(index=df_deaths.index)

for col in df_deaths.columns:
    # UF -> capital
    uf = capitals_dict[col]  # ex.: "Porto Velho" → "RO"

    pop_key = f"pop_{uf}"    # ex.: "pop_RO"

    # Normalization
    df_per100k[col] = (df_deaths[col] / pop_dict[pop_key] * 100000)

df_per100k



In [ ]:
# Deaths per Month Anomalies

monthly_means = df_per100k.groupby(df_per100k.index.month).mean()

df_anom = df_per100k.copy().astype(float)

for i in range(1, 13):  # meses de 1 a 12
    df_anom.loc[df_per100k.index.month == i] -= monthly_means.loc[i]

display(df_anom)



# Correlation with deaths - summer / winter

In [ ]:
capitals_list = [
                "Porto Velho",
                "Rio Branco",
                "Manaus",
                "Boa Vista",
                "Belém",
                "Macapá",
                "Palmas",

                "São Luís",
                "Teresina",
                "Fortaleza",
                "Natal",
                "João Pessoa",
                "Recife",
                "Maceió",
                "Aracaju",
                "Salvador",

                "Belo Horizonte",
                "Vitória",
                "Rio de Janeiro",
                "São Paulo",

                "Curitiba",
                "Florianópolis",
                "Porto Alegre",

                "Campo Grande",
                "Cuiabá",
                "Goiânia",
                "Brasília",
                ]

capitals_dict = {
                 "Porto Velho": "RO",
                 "Rio Branco": "AC",
                 "Manaus": "AM",
                 "Boa Vista": "RR",
                 "Belém": "PA",
                 "Macapá": "AP",
                 "Palmas": "TO",

                 "São Luís": "MA",
                 "Teresina": "PI",
                 "Fortaleza": "CE",
                 "Natal": "RN",
                 "João Pessoa": "PB",
                 "Recife": "PE",
                 "Maceió": "AL",
                 "Aracaju": "SE",
                 "Salvador": "BA",

                 "Belo Horizonte": "MG",
                 "Vitória": "ES",
                 "Rio de Janeiro": "RJ",
                 "São Paulo": "SP",

                 "Curitiba": "PR",
                 "Florianópolis": "SC",
                 "Porto Alegre": "RS",

                 "Campo Grande": "MS",
                 "Cuiabá": "MT",
                 "Goiânia": "GO",
                 "Brasília": "DF",
                 }

pop_dict = {
           "pop_RO": 460434,     # Porto Velho
           "pop_AC": 364756,     # Rio Branco
           "pop_AM": 2063689,    # Manaus
           "pop_RR": 413486,     # Boa Vista
           "pop_PA": 1303403,    # Belém
           "pop_AP": 442933,     # Macapá
           "pop_TO": 302692,     # Palmas

           "pop_MA": 1037775,    # São Luís
           "pop_PI": 866300,     # Teresina
           "pop_CE": 2428708,    # Fortaleza
           "pop_RN": 751300,     # Natal
           "pop_PB": 833932,     # João Pessoa
           "pop_PE": 1488920,    # Recife
           "pop_AL": 957916,     # Maceió
           "pop_SE": 602757,     # Aracaju
           "pop_BA": 2417678,    # Salvador

           "pop_MG": 2315560,    # Belo Horizonte
           "pop_ES": 322869,     # Vitória
           "pop_RJ": 6211223,    # Rio de Janeiro
           "pop_SP": 11451999,   # São Paulo

           "pop_PR": 1773718,    # Curitiba
           "pop_SC": 537211,     # Florianópolis
           "pop_RS": 1332845,    # Porto Alegre

           "pop_MS": 898100,     # Campo Grande
           "pop_MT": 650877,     # Cuiabá
           "pop_GO": 1437366,    # Goiânia
           "pop_DF": 2817381     # Brasília
           }

base_cols = [
            "Ind_Prod_Monthly",
            "ind_prod_monthly_qtd",
            "ind_prod_3episodes",
            "ind_prod_5episodes",
            "noaa_qtd_monthly",
            "noaa_3episodes",
            "noaa_5episodes",
            "omm_qtd_monthly",
            "omm_3episodes",
            "omm_5episodes",
            ]



In [ ]:
pth = str(CLUSTERING_METRICS_FILE)
df_hi = pd.read_parquet(pth)
df_hi = df_hi.loc['1999-01-01':'2023-12-01']
df_hi



In [ ]:
# Deaths
path_deaths = str(HEALTH_DEATHS_FILE)
df_deaths = pd.read_csv(path_deaths, sep=';')
df_deaths['time'] = pd.to_datetime(df_deaths['time'])
df_deaths.set_index('time', inplace=True)
df_deaths.index = pd.date_range(start="1996-01-01", periods=len(df_deaths), freq="MS")

df_per100k = pd.DataFrame(index=df_deaths.index)

for col in df_deaths.columns:
    # UF -> capital
    uf = capitals_dict[col]  # ex.: "Porto Velho" → "RO"

    pop_key = f"pop_{uf}"    # ex.: "pop_RO"

    # Normalization
    df_per100k[col] = (df_deaths[col] / pop_dict[pop_key] * 100000)

df_per100k



In [ ]:
# Deaths per Month Anomalies

monthly_means = df_per100k.groupby(df_per100k.index.month).mean()

df_anom = df_per100k.copy().astype(float)

for i in range(1, 13):  # meses de 1 a 12
    df_anom.loc[df_per100k.index.month == i] -= monthly_means.loc[i]

df_anom = df_anom.loc['1999-01-01':'2023-12-01']
display(df_anom)



In [ ]:
output_path = str(RESULTS_DIR / "health_correlations")
os.makedirs(output_path, exist_ok=True)

ufs_order = list(capitals_dict.values())

season_list = [
               [12, 1, 2],
               [3, 4, 5],
               [6, 7, 8],
               [9, 10, 11],

               [10, 11, 12, 1, 2, 3],
               [4, 5, 6, 7, 8, 9]
              ]

season_name = ['DJF', 'MAM', 'JJA', 'SON', 'summer', 'winter']

for season, name_season in zip(season_list, season_name):

    idx = df_anom.index.month.isin(season)

    deaths_m = df_anom.loc[idx]
    hi_m = df_hi.loc[idx]

    df_corr = pd.DataFrame(index=ufs_order, columns=base_cols, dtype=float)

    for capital, uf in capitals_dict.items():

        deaths_series = deaths_m[capital]

        for base in base_cols:

            colname = f"{base}_{uf}"
            hi_series = hi_m[colname]

            df_pair = pd.concat([deaths_series, hi_series], axis=1).dropna()

            rho = df_pair.iloc[:, 0].corr(
                df_pair.iloc[:, 1],
                method="spearman"
            )

            df_corr.loc[uf, base] = rho

    df_corr.to_csv(
                   f"{output_path}/corr_season_{name_season}.csv",
                   encoding="utf-8"
                  )

    print(f"Salvo corr_season_{name_season}.csv")




# Hospitalizations

In [ ]:
capitals_list = [
                "Porto Velho",
                "Rio Branco",
                "Manaus",
                "Boa Vista",
                "Belém",
                "Macapá",
                "Palmas",

                "São Luís",
                "Teresina",
                "Fortaleza",
                "Natal",
                "João Pessoa",
                "Recife",
                "Maceió",
                "Aracaju",
                "Salvador",

                "Belo Horizonte",
                "Vitória",
                "Rio de Janeiro",
                "São Paulo",

                "Curitiba",
                "Florianópolis",
                "Porto Alegre",

                "Campo Grande",
                "Cuiabá",
                "Goiânia",
                "Brasília",
                ]

capitals_dict = {
                 "Porto Velho": "RO",
                 "Rio Branco": "AC",
                 "Manaus": "AM",
                 "Boa Vista": "RR",
                 "Belém": "PA",
                 "Macapá": "AP",
                 "Palmas": "TO",

                 "São Luís": "MA",
                 "Teresina": "PI",
                 "Fortaleza": "CE",
                 "Natal": "RN",
                 "João Pessoa": "PB",
                 "Recife": "PE",
                 "Maceió": "AL",
                 "Aracaju": "SE",
                 "Salvador": "BA",

                 "Belo Horizonte": "MG",
                 "Vitória": "ES",
                 "Rio de Janeiro": "RJ",
                 "São Paulo": "SP",

                 "Curitiba": "PR",
                 "Florianópolis": "SC",
                 "Porto Alegre": "RS",

                 "Campo Grande": "MS",
                 "Cuiabá": "MT",
                 "Goiânia": "GO",
                 "Brasília": "DF",
                 }

pop_dict = {
           "pop_RO": 460434,     # Porto Velho
           "pop_AC": 364756,     # Rio Branco
           "pop_AM": 2063689,    # Manaus
           "pop_RR": 413486,     # Boa Vista
           "pop_PA": 1303403,    # Belém
           "pop_AP": 442933,     # Macapá
           "pop_TO": 302692,     # Palmas

           "pop_MA": 1037775,    # São Luís
           "pop_PI": 866300,     # Teresina
           "pop_CE": 2428708,    # Fortaleza
           "pop_RN": 751300,     # Natal
           "pop_PB": 833932,     # João Pessoa
           "pop_PE": 1488920,    # Recife
           "pop_AL": 957916,     # Maceió
           "pop_SE": 602757,     # Aracaju
           "pop_BA": 2417678,    # Salvador

           "pop_MG": 2315560,    # Belo Horizonte
           "pop_ES": 322869,     # Vitória
           "pop_RJ": 6211223,    # Rio de Janeiro
           "pop_SP": 11451999,   # São Paulo

           "pop_PR": 1773718,    # Curitiba
           "pop_SC": 537211,     # Florianópolis
           "pop_RS": 1332845,    # Porto Alegre

           "pop_MS": 898100,     # Campo Grande
           "pop_MT": 650877,     # Cuiabá
           "pop_GO": 1437366,    # Goiânia
           "pop_DF": 2817381     # Brasília
           }

base_cols = [
            "Ind_Prod_Monthly",
            "ind_prod_monthly_qtd",
            "ind_prod_3episodes",
            "ind_prod_5episodes",
            "noaa_qtd_monthly",
            "noaa_3episodes",
            "noaa_5episodes",
            "omm_qtd_monthly",
            "omm_3episodes",
            "omm_5episodes",
            ]



In [ ]:
pth = str(CLUSTERING_METRICS_FILE)
df_hi = pd.read_parquet(pth)
df_hi = df_hi.loc['1999-01-01':'2023-12-01']
df_hi



In [ ]:
# Hospitalizations
path_deaths = str(HEALTH_HOSPITALIZATIONS_FILE)
df_deaths = pd.read_csv(path_deaths, sep=';')
df_deaths['time'] = pd.to_datetime(df_deaths['time'])
df_deaths.set_index('time', inplace=True)
df_deaths.index = pd.date_range(start="1996-01-01", periods=len(df_deaths), freq="MS")

df_per100k = pd.DataFrame(index=df_deaths.index)

for col in df_deaths.columns:
    # UF -> capital
    uf = capitals_dict[col]  # ex.: "Porto Velho" → "RO"

    pop_key = f"pop_{uf}"    # ex.: "pop_RO"

    # Normalization
    df_per100k[col] = (df_deaths[col] / pop_dict[pop_key] * 100000)

df_per100k



In [ ]:
# Hospitalizations per Month Anomalies

monthly_means = df_per100k.groupby(df_per100k.index.month).mean()

df_anom = df_per100k.copy().astype(float)

for i in range(1, 13):  # meses de 1 a 12
    df_anom.loc[df_per100k.index.month == i] -= monthly_means.loc[i]

df_anom = df_anom.loc['1999-01-01':'2023-12-01']
display(df_anom)



In [ ]:
output_path = str(RESULTS_DIR / "health_correlations")
os.makedirs(output_path, exist_ok=True)

ufs_order = list(capitals_dict.values())

season_list = [
               [12, 1, 2],
               [3, 4, 5],
               [6, 7, 8],
               [9, 10, 11],

               [10, 11, 12, 1, 2, 3],
               [4, 5, 6, 7, 8, 9]
              ]

season_name = ['DJF', 'MAM', 'JJA', 'SON', 'summer', 'winter']

for season, name_season in zip(season_list, season_name):

    idx = df_anom.index.month.isin(season)

    deaths_m = df_anom.loc[idx]
    hi_m = df_hi.loc[idx]

    df_corr = pd.DataFrame(index=ufs_order, columns=base_cols, dtype=float)

    for capital, uf in capitals_dict.items():

        deaths_series = deaths_m[capital]

        for base in base_cols:

            colname = f"{base}_{uf}"
            hi_series = hi_m[colname]

            df_pair = pd.concat([deaths_series, hi_series], axis=1).dropna()

            rho = df_pair.iloc[:, 0].corr(
                df_pair.iloc[:, 1],
                method="spearman"
            )

            df_corr.loc[uf, base] = rho

    df_corr.to_csv(
                   f"{output_path}/corr_hospitalizations_season_{name_season}.csv",
                   encoding="utf-8"
                  )

    print(f"Salvo corr_hospitalizations_season_{name_season}.csv")


